# 🩺 Medical RAG Assistant using LangChain, FAISS & OpenAI

## Project Overview

This project implements a Retrieval-Augmented Generation (RAG) system for answering questions from a medical document.

### Objectives

- Load a medical PDF document
- Split the document into semantic chunks
- Generate embeddings using Hugging Face
- Store embeddings in a FAISS vector database
- Retrieve relevant context based on user queries
- Generate accurate answers using OpenAI GPT
- Evaluate retrieval performance using similarity scores and response times

### Tech Stack

- Python
- Google Colab
- LangChain
- FAISS
- Hugging Face Embeddings
- OpenAI GPT

# 1️⃣ Install Required Libraries

Install all dependencies required for building the Retrieval-Augmented Generation (RAG) pipeline.

In [27]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
faiss-cpu \
sentence-transformers \
pypdf \
openai

# 2️⃣ Import Libraries & Configure OpenAI

Import the required Python libraries and securely load the OpenAI API key from Google Colab Secrets.

In [28]:
import time

from google.colab import userdata

from openai import OpenAI

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

In [29]:
client = OpenAI(
    api_key=userdata.get("rag_key")
)

# 3️⃣ Upload Medical PDF

Upload the medical document that will serve as the knowledge base for the RAG system.

In [30]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

Saving Medicine - Wikipedia copy.pdf to Medicine - Wikipedia copy (1).pdf


# 4️⃣ Load PDF Document

Load the uploaded PDF into LangChain documents using the PyPDFLoader.

In [31]:
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(len(documents))

30


# 5️⃣ Preview Document

Display a sample page from the uploaded document to verify that it has been loaded correctly.

In [32]:
print(documents[0].page_content)

The Doctor by Sir Luke Fildes (1891)
Medicine
Medicine i s t h e science[1] a n d
practice[2] of  caring  for  patients,
managing  the  diagnosis, prognosis,
prevention, treatment and palliation
of  their  injury o r disease, w h i l e
promoting  their  health.  Medicine
encompasses a variety of health care
practices  which  evolved  to  maintain
and  restore  health through  the
prevention a n d t r e a t m e n t o f illness
and  infection. C o n t e m p o r a r y
medicine applies biomedical sciences,
biomedical  research, genetics, a n d
medical technology to diagnose, treat,
and  prevent  injury  and  disease,
typically  through  various
pharmaceuticals or surgery, but also through therapies such as psychotherapy, external splints and
traction, medical devices, biologics, and ionizing radiation, amongst others.[3]
Medicine has been practiced since prehistoric times, and for most of this time it was an art (an area
of creativity and skill), frequently having connections to the religi

# 6️⃣ Split Document into Chunks

Split the document into overlapping text chunks to improve retrieval accuracy and preserve context.

In [33]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 182


# 7️⃣ Preview Text Chunks

Display sample chunks to verify the chunking strategy.

In [34]:
for i in range(3):
    print("="*60)
    print("Chunk", i+1)
    print(chunks[i].page_content)

Chunk 1
The Doctor by Sir Luke Fildes (1891)
Medicine
Medicine i s t h e science[1] a n d
practice[2] of  caring  for  patients,
managing  the  diagnosis, prognosis,
prevention, treatment and palliation
of  their  injury o r disease, w h i l e
promoting  their  health.  Medicine
encompasses a variety of health care
practices  which  evolved  to  maintain
and  restore  health through  the
prevention a n d t r e a t m e n t o f illness
and  infection. C o n t e m p o r a r y
medicine applies biomedical sciences,
biomedical  research, genetics, a n d
medical technology to diagnose, treat,
and  prevent  injury  and  disease,
typically  through  various
Chunk 2
biomedical  research, genetics, a n d
medical technology to diagnose, treat,
and  prevent  injury  and  disease,
typically  through  various
pharmaceuticals or surgery, but also through therapies such as psychotherapy, external splints and
traction, medical devices, biologics, and ionizing radiation, amongst others.[3]
Medicine has b

In [35]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# 9️⃣ Build FAISS Vector Database

Create a FAISS vector index from the generated embeddings for efficient semantic search.

In [36]:
vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)
print("Vector database created successfully.")

# 🔟 Save Vector Database

Save the FAISS index locally for future reuse.

In [37]:
vector_db.save_local("faiss_index")

# 1️⃣1️⃣ Load Vector Database

Reload the saved FAISS index to verify persistence.

In [38]:
vector_db = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

# 1️⃣2️⃣ Generate Sample Query Embedding

Convert a sample query into an embedding vector.

In [39]:
vector = embedding_model.embed_query("What is Medicine")

print(len(vector))

384


# 1️⃣3️⃣ Perform Similarity Search

Retrieve the most relevant document chunks from the FAISS vector database.

In [43]:
query = "What is Medicine?"

results = vector_db.similarity_search(query, k=5)

# 1️⃣4️⃣ Display Retrieved Chunks

Display the retrieved chunks along with similarity scores.

In [44]:
for i, doc in enumerate(results):

    print("="*60)

    print("Retrieved Chunk", i+1)

    print(doc.page_content)

Retrieved Chunk 1
stitched.
Prescientific  forms  of  medicine,  now  known  as  traditional  medicine o r folk  medicine, r e m a i n
commonly used in the absence of scientific medicine and are thus called alternative medicine.
Alternative treatments outside of scientific medicine with ethical, safety and efficacy concerns are
termed quackery or being based on fringe science.[4]
Medicine (UK: /ˈmɛdsɪn/ ⡑, US: /ˈmɛdɪsɪn/ ⡑) is the science and practice of the diagnosis,
prognosis, treatment, and prevention of disease.[5][6] The word "medicine" is derived from Latin
Etymology
Medicine - Wikipedia https://en.wikipedia.org/wiki/Medicine
1 of 30 27/07/26, 6:38 pm
Retrieved Chunk 2
biomedical  research, genetics, a n d
medical technology to diagnose, treat,
and  prevent  injury  and  disease,
typically  through  various
pharmaceuticals or surgery, but also through therapies such as psychotherapy, external splints and
traction, medical devices, biologics, and ionizing radiation, amongst other

# 1️⃣5️⃣ Define Retrieval Function

Create a reusable function that retrieves the most relevant document context for a given question.

In [45]:
# Retreiver Context
def retrieve_context(question, k=5):
    results = vector_db.similarity_search_with_score(question, k=k)

    docs = [doc for doc, score in results]
    scores = [score for doc, score in results]

    context = "\n\n".join(doc.page_content for doc in docs)

    return context, docs, scores

# 1️⃣6️⃣ Define Answer Generation Function

Generate answers using OpenAI GPT based only on the retrieved document context.

In [46]:
#Generate_Answers


def generate_answer(question, context):

    prompt = f"""
You are a medical assistant.

Answer ONLY using the retrieved context.

If the answer is not found in the context, reply:

"I couldn't find that information in the uploaded document."

Do not use outside knowledge.

Context:

{context}

Question:

{question}

Answer:
"""

    response = client.responses.create(
        model="gpt-5-nano",
        input=prompt
    )

    return response.output_text

# 1️⃣7️⃣ Define Main RAG Pipeline

Combine retrieval and answer generation into a single function that also computes evaluation metrics.

In [47]:
#Ask_Question

def ask_question(question):

    # Measure retrieval time
    retrieval_start = time.time()

    context, docs, scores = retrieve_context(question)

    retrieval_time = time.time() - retrieval_start

    # Measure generation time
    generation_start = time.time()

    answer = generate_answer(question, context)

    generation_time = time.time() - generation_start

    avg_similarity = round(
        sum(float(score) for score in scores) / len(scores),
        3
    )

    return {
        "Question": question,
        "Answer": answer,
        "Retrieved Chunks": len(docs),
        "Average Similarity": avg_similarity,
        "Retrieval Time (sec)": round(retrieval_time, 2),
        "Generation Time (sec)": round(generation_time, 2)
    }

# 1️⃣8️⃣ Test Context Retrieval

Verify that the retrieval function returns the expected context.

In [48]:
question = "What is medicine?"
context = retrieve_context(question)
print(context)

('stitched.\nPrescientific  forms  of  medicine,  now  known  as  traditional  medicine o r folk  medicine, r e m a i n\ncommonly used in the absence of scientific medicine and are thus called alternative medicine.\nAlternative treatments outside of scientific medicine with ethical, safety and efficacy concerns are\ntermed quackery or being based on fringe science.[4]\nMedicine (UK: /ˈmɛdsɪn/ ⡑, US: /ˈmɛdɪsɪn/ ⡑) is the science and practice of the diagnosis,\nprognosis, treatment, and prevention of disease.[5][6] The word "medicine" is derived from Latin\nEtymology\nMedicine - Wikipedia https://en.wikipedia.org/wiki/Medicine\n1 of 30 27/07/26, 6:38 pm\n\nbiomedical  research, genetics, a n d\nmedical technology to diagnose, treat,\nand  prevent  injury  and  disease,\ntypically  through  various\npharmaceuticals or surgery, but also through therapies such as psychotherapy, external splints and\ntraction, medical devices, biologics, and ionizing radiation, amongst others.[3]\nMedicine h

# 1️⃣9️⃣ Test Answer Generation

Verify that the language model generates an answer using the retrieved context.

In [49]:
answer = generate_answer(question,context)
print(answer)

Medicine is the science and practice of the diagnosis, prognosis, treatment, and prevention of disease.


# 2️⃣0️⃣ Evaluate the RAG System

Run multiple test questions and display:

- User Question
- Generated Answer
- Retrieved Chunks
- Average Similarity Score
- Retrieval Time
- Generation Time

In [51]:
test_questions = [
    "What is medicine?",
    "What is traditional medicine?",
    "What is medical ethics?",
    "What is diagnosis?",
    "What is prognosis?"
]

for i, question in enumerate(test_questions, start=1):

    result = ask_question(question)

    print("=" * 70)
    print(f"🧪 Test Case {i}")
    print("=" * 70)

    print(f"❓ Question : {result['Question']}\n")
    print(f"💡 Answer :\n{result['Answer']}\n")

    print("📊 Evaluation")
    print(f"• Retrieved Chunks    : {result['Retrieved Chunks']}")
    print(f"• Average Similarity  : {result['Average Similarity']}")
    print(f"• Retrieval Time      : {result['Retrieval Time (sec)']} sec")
    print(f"• Generation Time     : {result['Generation Time (sec)']} sec")

    print("\n")

🧪 Test Case 1
❓ Question : What is medicine?

💡 Answer :
Medicine is the science and practice of the diagnosis, prognosis, treatment, and prevention of disease.

📊 Evaluation
• Retrieved Chunks    : 5
• Average Similarity  : 0.745
• Retrieval Time      : 0.13 sec
• Generation Time     : 3.99 sec


🧪 Test Case 2
❓ Question : What is traditional medicine?

💡 Answer :
Traditional medicine, or folk medicine, refers to prescientific forms of medicine that remain commonly used in the absence of scientific medicine and are thus called alternative medicine.

📊 Evaluation
• Retrieved Chunks    : 5
• Average Similarity  : 0.801
• Retrieval Time      : 0.02 sec
• Generation Time     : 4.06 sec


🧪 Test Case 3
❓ Question : What is medical ethics?

💡 Answer :
Medical ethics is a system of moral principles that apply values and judgments to the practice of medicine. As a scholarly discipline, medical ethics encompasses its practical application in clinical settings as well as work on its history, ph

# ✅ Project Summary

This project demonstrates an end-to-end Retrieval-Augmented Generation (RAG) system for answering questions from medical documents.

## Workflow

Medical PDF

⬇

Document Loading

⬇

Text Chunking

⬇

Embedding Generation

⬇

FAISS Vector Database

⬇

Semantic Retrieval

⬇

OpenAI GPT

⬇

Final Answer

## Features

- Semantic document retrieval
- FAISS vector search
- Hugging Face embeddings
- OpenAI GPT integration
- Evaluation metrics
  - Retrieved chunks
  - Similarity score
  - Retrieval time
  - Generation time

## Future Improvements

- Hybrid Search (FAISS + BM25)
- Cross-Encoder Reranking
- User Feedback Collection
- Query Logging
- Live Performance Dashboard